# Validation baselines and QLoRA

This notebook runs the validation-only prompting comparison, checks completion-only loss masking, trains one SmolLM3 QLoRA adapter, reloads it, and saves validation predictions. The final test file is not loaded here. In Kaggle, turn Internet on before running the setup cell; the setup fetches only the four required source and data files from the public GitHub repository.

In [2]:
!pip -q install 'transformers==5.10.1' 'peft==0.20.0' 'trl==0.29.0' bitsandbytes accelerate datasets 'mlflow==3.15.1' scikit-learn

from pathlib import Path
from urllib.error import URLError
from urllib.request import urlopen

REPO_REF = '525858b'  # frozen GitHub revision containing frozen_full_v2 data
RAW_BASE = f'https://raw.githubusercontent.com/goyashek/civic-grievance-structurer/{REPO_REF}'
ROOT = Path('/kaggle/working/civicstruct')
REQUIRED_FILES = ('src/evaluate.py', 'src/schema.py', 'data/surface_variants.jsonl', 'data/public_training_examples.jsonl')
try:
    for relative_path in REQUIRED_FILES:
        destination = ROOT / relative_path
        destination.parent.mkdir(parents=True, exist_ok=True)
        with urlopen(f'{RAW_BASE}/{relative_path}', timeout=60) as response:
            destination.write_bytes(response.read())
except URLError as exc:
    raise RuntimeError('Turn on Kaggle Internet to fetch the frozen project files from GitHub.') from exc
print({'source': RAW_BASE, 'fetched': list(REQUIRED_FILES), 'test_loaded': False})

{'source': 'https://raw.githubusercontent.com/goyashek/civic-grievance-structurer/525858b', 'fetched': ['src/evaluate.py', 'src/schema.py', 'data/surface_variants.jsonl', 'data/public_training_examples.jsonl'], 'test_loaded': False}


In [3]:
import gc
import enum
import json
import math
import os
import platform
import random
import shutil
import sys
import time
from importlib.metadata import version
from pathlib import Path

ROOT = Path('/kaggle/working/civicstruct')
sys.path.insert(0, str(ROOT))
import mlflow
import torch
from datasets import Dataset
from peft import LoraConfig, PeftModel
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

from src.evaluate import evaluate_outputs
def json_default(value):
    if isinstance(value, set):
        return sorted(value, key=str)
    if isinstance(value, (Path, os.PathLike)):
        return os.fspath(value)
    if isinstance(value, torch.dtype):
        return str(value)
    if isinstance(value, enum.Enum):
        return value.value
    if hasattr(value, 'item'):
        try:
            return value.item()
        except (TypeError, ValueError):
            pass
    if hasattr(value, 'tolist'):
        return value.tolist()
    return str(value)
def prepare_trainable_parameters(model):
    if hasattr(model, 'enable_input_require_grads'):
        model.enable_input_require_grads()
    for parameter in model.parameters():
        if parameter.requires_grad and parameter.dtype != torch.float32:
            parameter.data = parameter.data.float()
    assert all(parameter.dtype == torch.float32 for parameter in model.parameters() if parameter.requires_grad)

SEED = 42
MODEL_NAME = 'HuggingFaceTB/SmolLM3-3B'
MODEL_REVISION = 'main'
MAX_NEW_TOKENS = 256
MAX_LENGTH = 768
DEMO_COUNT = 3
BATCH_SIZE = 4
MODEL_DTYPE = torch.float16  # T4-safe; do not use BF16 on this runtime
OUTPUT = Path('/kaggle/working/civicstruct_output')
OUTPUT.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
torch.manual_seed(SEED)
assert torch.cuda.is_available(), 'Select a Colab GPU runtime before running this notebook.'
torch.cuda.manual_seed_all(SEED)
os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
mlflow.set_tracking_uri((OUTPUT / 'mlruns').as_uri())
mlflow.set_experiment('civicstruct-validation')
print({'device': torch.cuda.get_device_name(0), 'python': platform.python_version(), 'torch': torch.__version__})

{'device': 'Tesla T4', 'python': '3.12.13', 'torch': '2.10.0+cu128'}


In [4]:
def load_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

surfaces = load_jsonl(ROOT / 'data/surface_variants.jsonl')
public_training = load_jsonl(ROOT / 'data/public_training_examples.jsonl')
controlled_training = [row for row in surfaces if row['split'] == 'train']
validation = [row for row in surfaces if row['split'] == 'validation']
training = controlled_training + public_training
assert len(controlled_training) == 120
assert len(public_training) == 40
assert len(training) == 160
assert len(validation) == 50
assert len({row['surface_id'] for row in validation}) == len(validation)
assert not {row['case_id'] for row in training} & {row['case_id'] for row in validation}
assert len({row['case_id'] for row in training}) == len(training)
print({'training': len(training), 'validation': len(validation), 'test_loaded': False})

{'training': 160, 'validation': 50, 'test_loaded': False}


In [5]:
DOMAINS = ['public_transport', 'water_supply', 'sanitation_and_waste', 'roads_and_streetlights', 'electricity', 'welfare_or_document_service', 'other']
ISSUES = ['delay_or_non_arrival', 'service_outage_or_non_delivery', 'damaged_infrastructure', 'overcharging_or_payment_problem', 'record_or_document_error', 'staff_conduct', 'safety_or_health_hazard', 'other']
URGENCY = ['routine', 'time_sensitive', 'safety_critical']
MISSING = ['exact_location', 'date_or_time', 'service_identifier', 'transaction_or_reference_id', 'amount', 'supporting_evidence', 'affected_person_or_group', 'none']
SYSTEM_PROMPT = (
    'Structure one public-service complaint as exactly one JSON object. Use these fields in this order: '
    'service_domain, issue_type, location, event_date_or_time, amount_inr, service_identifier, urgency, missing_information, formal_summary. '
    f'Allowed service_domain values: {DOMAINS}. Allowed issue_type values: {ISSUES}. '
    f'Allowed urgency values: {URGENCY}. Allowed missing_information values: {MISSING}. '
    'Use null for absent scalar facts. Missing information must be a non-empty ordered list. Use ["none"] only when no important detail is missing. '
    'Do not guess facts. The formal summary must be one neutral sentence. Return no reasoning, markdown, or commentary.'
)

def messages_for(complaint, demos=()):
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}]
    for demo in demos:
        messages.extend([
            {'role': 'user', 'content': demo['complaint']},
            {'role': 'assistant', 'content': json.dumps(demo['gold'], ensure_ascii=False, separators=(',', ':'))},
        ])
    messages.append({'role': 'user', 'content': complaint})
    return messages

train_records = [
    {
        'prompt': messages_for(row['complaint']),
        'completion': [{'role': 'assistant', 'content': json.dumps(row['gold'], ensure_ascii=False, separators=(',', ':'))}],
    }
    for row in training
]
assert len(train_records) == 160
print(json.dumps(train_records[0], indent=2, ensure_ascii=False))

{
  "prompt": [
    {
      "role": "system",
      "content": "Structure one public-service complaint as exactly one JSON object. Use these fields in this order: service_domain, issue_type, location, event_date_or_time, amount_inr, service_identifier, urgency, missing_information, formal_summary. Allowed service_domain values: ['public_transport', 'water_supply', 'sanitation_and_waste', 'roads_and_streetlights', 'electricity', 'welfare_or_document_service', 'other']. Allowed issue_type values: ['delay_or_non_arrival', 'service_outage_or_non_delivery', 'damaged_infrastructure', 'overcharging_or_payment_problem', 'record_or_document_error', 'staff_conduct', 'safety_or_health_hazard', 'other']. Allowed urgency values: ['routine', 'time_sensitive', 'safety_critical']. Allowed missing_information values: ['exact_location', 'date_or_time', 'service_identifier', 'transaction_or_reference_id', 'amount', 'supporting_evidence', 'affected_person_or_group', 'none']. Use null for absent scalar fac

In [6]:
def rule_output(complaint):
    text = complaint.casefold()
    if any(word in text for word in ('water', 'tap ', 'tanker')):
        domain = 'water_supply'
    elif any(word in text for word in ('garbage', 'waste', 'sewage', 'sanitation', 'bin ', 'sweeper')):
        domain = 'sanitation_and_waste'
    elif any(word in text for word in ('electric', 'power', 'feeder', 'transformer')):
        domain = 'electricity'
    elif any(word in text for word in ('road', 'streetlight', 'street light', 'pothole', 'signal', 'sidewalk', 'pavement', 'parking meter')):
        domain = 'roads_and_streetlights'
    elif any(word in text for word in ('pension', 'certificate', 'benefit', 'welfare', 'application', 'document')):
        domain = 'welfare_or_document_service'
    elif any(word in text for word in ('bus', 'train', 'station', 'platform', 'ticket', 'route ')):
        domain = 'public_transport'
    else:
        domain = 'other'

    if any(word in text for word in ('charged', 'bill ', 'fee ', 'payment')):
        issue = 'overcharging_or_payment_problem'
    elif any(word in text for word in ('wrong', 'incorrect', 'marked completed', 'record ')):
        issue = 'record_or_document_error'
    elif any(word in text for word in ('shouted', 'insulted', 'rude', 'refused', 'mocked', 'ignored')):
        issue = 'staff_conduct'
    elif any(word in text for word in ('live wire', 'sparking', 'unsafe', 'hazard', 'collision', 'needles', 'sharp edges')):
        issue = 'safety_or_health_hazard'
    elif any(word in text for word in ('did not arrive', 'never arrived', 'never came', 'did not come', 'scheduled for', 'was due')):
        issue = 'delay_or_non_arrival'
    elif any(word in text for word in ('broken', 'cracked', 'damaged', 'pothole', 'leaking', 'raised sidewalk', 'leaning')):
        issue = 'damaged_infrastructure'
    elif any(word in text for word in ('no water', 'no power', 'unavailable', 'not working', 'has not worked', 'error for', 'no collection')):
        issue = 'service_outage_or_non_delivery'
    else:
        issue = 'other'

    urgency = 'safety_critical' if issue == 'safety_or_health_hazard' else ('time_sensitive' if any(word in text for word in ('for two days', 'for three days', 'for four days', 'for five days', 'for six days', 'for a week', 'since monday', 'since friday', 'since sunday')) else 'routine')
    return {
        'service_domain': domain, 'issue_type': issue, 'location': None, 'event_date_or_time': None,
        'amount_inr': None, 'service_identifier': None, 'urgency': urgency,
        'missing_information': ['exact_location'], 'formal_summary': complaint.strip(),
    }

rule_outputs = [json.dumps(rule_output(row['complaint']), separators=(',', ':')) for row in validation]
rule_scores = evaluate_outputs([row['gold'] for row in validation], rule_outputs)
print(rule_scores['strict'])

{'json_parse_count': 50, 'invalid_json_count': 0, 'schema_invalid_count': 0, 'schema_valid_count': 50, 'schema_validity_rate': 1.0, 'end_to_end_field_metrics': {'service_domain_macro_f1': 0.8827383796327896, 'issue_type_macro_f1': 0.8952380952380953, 'urgency_macro_f1': 0.7067395264116575, 'missing_information_macro_f1': 0.026785714285714284, 'location_normalized_match': 0.12, 'event_date_or_time_normalized_match': 0.12, 'amount_inr_exact_match': 0.92, 'service_identifier_exact_match': 0.28}, 'hallucinated_non_null_fields': {'count': 0, 'predicted_non_null_count': 0, 'rate': None}}


In [7]:
STATIC_IDS = ('canonical-001', 'canonical-020', 'canonical-030')
training_by_id = {row['case_id']: row for row in training}
static_demos = [training_by_id[case_id] for case_id in STATIC_IDS]
vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=1)
training_matrix = vectorizer.fit_transform([row['complaint'] for row in training])

def retrieve(complaint):
    scores = (training_matrix @ vectorizer.transform([complaint]).T).toarray().ravel()
    ranked = scores.argsort()[::-1]
    selected = [training[index] for index in ranked[:DEMO_COUNT]]
    assert len({row['case_id'] for row in selected}) == DEMO_COUNT
    return selected

retrieved = {row['surface_id']: retrieve(row['complaint']) for row in validation}
assert all(all(demo['split'] == 'train' for demo in demos) for demos in retrieved.values())
print({surface_id: [demo['case_id'] for demo in demos] for surface_id, demos in list(retrieved.items())[:3]})

{'canonical-121-formal-english': ['canonical-001', 'canonical-071', 'canonical-050'], 'canonical-121-informal-english': ['canonical-071', 'canonical-001', 'canonical-072'], 'canonical-122-formal-english': ['canonical-079', 'canonical-017', 'canonical-016']}


In [8]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=MODEL_DTYPE,
    bnb_4bit_use_double_quant=True,
)

def load_base():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
    tokenizer.padding_side = 'left'
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, revision=MODEL_REVISION, quantization_config=quantization_config,
        dtype=MODEL_DTYPE, device_map='auto'
    )
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.eos_token_id = tokenizer.eos_token_id
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    model.generation_config.eos_token_id = tokenizer.eos_token_id
    model.config.use_cache = True
    return tokenizer, model

def encoded_batch(tokenizer, message_batch):
    kwargs = dict(tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors='pt', padding=True, truncation=True, max_length=2048, enable_thinking=False)
    try:
        return tokenizer.apply_chat_template(message_batch, **kwargs)
    except TypeError:
        kwargs.pop('enable_thinking')
        return tokenizer.apply_chat_template(message_batch, **kwargs)

def generate_rows(tokenizer, model, rows, demo_lookup=None):
    outputs = []
    model.eval()
    for start in range(0, len(rows), BATCH_SIZE):
        batch = rows[start:start + BATCH_SIZE]
        messages = [messages_for(row['complaint'], () if demo_lookup is None else demo_lookup[row['surface_id']]) for row in batch]
        inputs = encoded_batch(tokenizer, messages).to(next(model.parameters()).device)
        prompt_lengths = inputs['attention_mask'].sum(dim=1).tolist()
        padded_length = inputs['input_ids'].shape[1]
        torch.cuda.synchronize()
        started = time.perf_counter()
        with torch.inference_mode():
            generated = model.generate(**inputs, do_sample=False, max_new_tokens=MAX_NEW_TOKENS, use_cache=True, pad_token_id=tokenizer.pad_token_id)
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - started
        responses = tokenizer.batch_decode(generated[:, padded_length:], skip_special_tokens=True)
        outputs.extend([
            {'surface_id': row['surface_id'], 'case_id': row['case_id'], 'response': response.strip(), 'latency_seconds': elapsed / len(batch), 'prompt_tokens': prompt_length}
            for row, response, prompt_length in zip(batch, responses, prompt_lengths)
        ])
    return outputs

tokenizer, base_model = load_base()
MODEL_COMMIT = getattr(base_model.config, '_commit_hash', None) or MODEL_REVISION
print({'model': MODEL_NAME, 'revision': MODEL_COMMIT, 'parameters': base_model.num_parameters()})

{'model': 'HuggingFaceTB/SmolLM3-3B', 'revision': 'a07cc9a04f16550a088caea529712d1d335b0ac1', 'parameters': 3075098624}


In [9]:
def save_json(path, value):
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False, default=json_default), encoding='utf-8')

baseline_runs = {}
methods = {
    'deterministic_rules': [{'surface_id': row['surface_id'], 'case_id': row['case_id'], 'response': raw, 'latency_seconds': 0.0, 'prompt_tokens': 0} for row, raw in zip(validation, rule_outputs)],
    'zero_shot': generate_rows(tokenizer, base_model, validation),
    'static_few_shot': generate_rows(tokenizer, base_model, validation, {row['surface_id']: static_demos for row in validation}),
    'retrieved_few_shot': generate_rows(tokenizer, base_model, validation, retrieved),
}
for method, outputs in methods.items():
    scores = evaluate_outputs([row['gold'] for row in validation], [item['response'] for item in outputs])
    record = {
        'method': method, 'model_name': None if method == 'deterministic_rules' else MODEL_NAME,
        'model_revision': None if method == 'deterministic_rules' else MODEL_COMMIT,
        'decoding': None if method == 'deterministic_rules' else {'do_sample': False, 'max_new_tokens': MAX_NEW_TOKENS},
        'mean_latency_seconds': sum(item['latency_seconds'] for item in outputs) / len(outputs),
        'mean_prompt_tokens': sum(item['prompt_tokens'] for item in outputs) / len(outputs),
        'demonstration_count': 0 if method in ('deterministic_rules', 'zero_shot') else DEMO_COUNT,
        'scores': scores, 'outputs': outputs,
    }
    if method == 'retrieved_few_shot':
        record['retrieved_case_ids'] = {surface_id: [demo['case_id'] for demo in demos] for surface_id, demos in retrieved.items()}
    with mlflow.start_run(run_name=method) as run:
        mlflow.log_params({'method': method, 'model_name': record['model_name'] or 'none', 'model_revision': record['model_revision'] or 'none', 'demonstration_count': record['demonstration_count']})
        mlflow.log_metrics({'schema_validity_rate': scores['strict']['schema_validity_rate'], 'mean_latency_seconds': record['mean_latency_seconds'], 'mean_prompt_tokens': record['mean_prompt_tokens']})
        mlflow.log_text(json.dumps(record, indent=2, default=json_default), 'validation_run.json')
        record['mlflow_run_id'] = run.info.run_id
    baseline_runs[method] = record
save_json(OUTPUT / 'validation_baselines.json', baseline_runs)
print({method: {'schema_validity': run['scores']['strict']['schema_validity_rate'], 'latency': round(run['mean_latency_seconds'], 3)} for method, run in baseline_runs.items()})

{'deterministic_rules': {'schema_validity': 1.0, 'latency': 0.0}, 'zero_shot': {'schema_validity': 0.0, 'latency': 2.668}, 'static_few_shot': {'schema_validity': 0.74, 'latency': 1.85}, 'retrieved_few_shot': {'schema_validity': 0.88, 'latency': 1.967}}


## QLoRA smoke test and full run

The smoke run checks trainable parameters, completion-only masking, finite loss, adapter save, and clean reload. The full run then starts from a fresh base model.

In [10]:
del base_model
gc.collect()
torch.cuda.empty_cache()
lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM', target_modules='all-linear')
smoke_dir = OUTPUT / 'smoke_adapter'
tokenizer, smoke_base = load_base()
smoke_base.config.use_cache = False
smoke_args = SFTConfig(
    output_dir=str(OUTPUT / 'smoke_checkpoints'), max_steps=6, per_device_train_batch_size=1, gradient_accumulation_steps=2,
    learning_rate=2e-4, logging_steps=1, save_strategy='no', report_to='none', fp16=False, bf16=False, max_length=MAX_LENGTH,
    completion_only_loss=True, gradient_checkpointing=True, seed=SEED,
)
smoke_trainer = SFTTrainer(
    model=smoke_base, args=smoke_args, train_dataset=Dataset.from_list(train_records[:16]),
    processing_class=tokenizer, peft_config=lora_config,
)
prepare_trainable_parameters(smoke_trainer.model)
batch = next(iter(smoke_trainer.get_train_dataloader()))
masked_tokens = int((batch['labels'] == -100).sum())
trained_tokens = int((batch['labels'] != -100).sum())
assert masked_tokens > 0 and trained_tokens > 0
trainable = sum(parameter.numel() for parameter in smoke_trainer.model.parameters() if parameter.requires_grad)
total = sum(parameter.numel() for parameter in smoke_trainer.model.parameters())
assert 0 < trainable < total
smoke_result = smoke_trainer.train()
losses = [item['loss'] for item in smoke_trainer.state.log_history if 'loss' in item]
assert losses and all(math.isfinite(loss) for loss in losses) and min(losses[1:] or losses) < losses[0]
smoke_trainer.save_model(smoke_dir)
smoke_metadata = {
    'steps': smoke_trainer.state.global_step, 'losses': losses, 'masked_tokens': masked_tokens, 'trained_tokens': trained_tokens,
    'trainable_parameters': trainable, 'total_parameters': total, 'trainable_percent': 100 * trainable / total,
    'metrics': smoke_result.metrics,
}
del smoke_trainer, smoke_base
gc.collect()
torch.cuda.empty_cache()
reload_tokenizer, reload_base = load_base()
reloaded_smoke = PeftModel.from_pretrained(reload_base, smoke_dir)
smoke_rows = validation[:2]
smoke_after = generate_rows(reload_tokenizer, reloaded_smoke, smoke_rows)
smoke_before = [next(item for item in baseline_runs['zero_shot']['outputs'] if item['surface_id'] == row['surface_id']) for row in smoke_rows]
smoke_metadata['reload_check'] = {'before': smoke_before, 'after': smoke_after}
save_json(OUTPUT / 'smoke_run.json', smoke_metadata)
print(smoke_metadata)
del reloaded_smoke, reload_base
gc.collect()
torch.cuda.empty_cache()

{'steps': 6, 'losses': [0.5637710690498352, 0.23015564680099487, 0.10279547423124313, 0.319186270236969, 0.12363345175981522, 0.07578067481517792], 'masked_tokens': 315, 'trained_tokens': 82, 'trainable_parameters': 30228480, 'total_parameters': 1699186688, 'trainable_percent': 1.778996987999002, 'metrics': {'train_runtime': 12.3941, 'train_samples_per_second': 0.968, 'train_steps_per_second': 0.484, 'total_flos': 80060642893824.0, 'train_loss': 0.23588709781567255}, 'reload_check': {'before': [{'surface_id': 'canonical-121-formal-english', 'case_id': 'canonical-121', 'response': '```json\n{\n  "service_domain": "public_transport",\n  "issue_type": "delay_or_non_arrival",\n  "location": "Hilltop stop",\n  "event_date_or_time": "Tuesday, 8:40",\n  "amount_inr": null,\n  "service_identifier": null,\n  "urgency": "routine",\n  "missing_information": ["exact_location", "date_or_time", "service_identifier"],\n  "formal_summary": "Route 44 did not arrive at Hilltop stop at Tuesday, 8:40."\n}

In [11]:
adapter_dir = OUTPUT / 'qlora_adapter'
tokenizer, full_base = load_base()
full_base.config.use_cache = False
torch.cuda.reset_peak_memory_stats()
full_args = SFTConfig(
    output_dir=str(OUTPUT / 'full_checkpoints'), num_train_epochs=1, per_device_train_batch_size=1, gradient_accumulation_steps=8,
    learning_rate=2e-4, warmup_ratio=0.03, lr_scheduler_type='cosine', logging_steps=1, save_strategy='no',
    report_to='none', fp16=False, bf16=False, max_length=MAX_LENGTH, completion_only_loss=True, gradient_checkpointing=True, seed=SEED,
)
trainer = SFTTrainer(
    model=full_base, args=full_args, train_dataset=Dataset.from_list(train_records), processing_class=tokenizer, peft_config=lora_config,
)
prepare_trainable_parameters(trainer.model)
started = time.perf_counter()
train_result = trainer.train()
training_seconds = time.perf_counter() - started
losses = [item['loss'] for item in trainer.state.log_history if 'loss' in item]
assert losses and all(math.isfinite(loss) for loss in losses)
trainer.save_model(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
peak_memory_mb = torch.cuda.max_memory_allocated() / 1024**2
training_metadata = {
    'model_name': MODEL_NAME, 'model_revision': MODEL_COMMIT, 'dataset_rows': len(training), 'dataset_version': 'frozen_full_v2',
    'configuration': full_args.to_dict(), 'lora': lora_config.to_dict(), 'losses': losses, 'train_metrics': train_result.metrics,
    'training_seconds': training_seconds, 'peak_gpu_memory_mb': peak_memory_mb, 'device': torch.cuda.get_device_name(0),
    'packages': {name: version(name) for name in ('torch', 'transformers', 'peft', 'trl', 'bitsandbytes', 'datasets', 'mlflow')},
}
with mlflow.start_run(run_name='smollm3-qlora-full') as run:
    mlflow.log_params({'model_name': MODEL_NAME, 'model_revision': MODEL_COMMIT, 'dataset_rows': len(training), 'epochs': 1, 'lora_r': 16, 'learning_rate': 2e-4})
    mlflow.log_metrics({'train_loss': train_result.metrics['train_loss'], 'training_seconds': training_seconds, 'peak_gpu_memory_mb': peak_memory_mb})
    training_metadata['mlflow_run_id'] = run.info.run_id
    mlflow.log_text(json.dumps(training_metadata, indent=2, default=json_default), 'training_metadata.json')
save_json(OUTPUT / 'qlora_training_metadata.json', training_metadata)
print({key: training_metadata[key] for key in ('model_revision', 'dataset_rows', 'training_seconds', 'peak_gpu_memory_mb', 'mlflow_run_id')})

{'model_revision': 'a07cc9a04f16550a088caea529712d1d335b0ac1', 'dataset_rows': 160, 'training_seconds': 163.93159194500004, 'peak_gpu_memory_mb': 1570.77685546875, 'mlflow_run_id': '29374850f4cc436a91a76f5f3cffd19f'}


In [12]:
del trainer, full_base
gc.collect()
torch.cuda.empty_cache()
reload_tokenizer, reload_base = load_base()
reloaded_model = PeftModel.from_pretrained(reload_base, adapter_dir)
qlora_outputs = generate_rows(reload_tokenizer, reloaded_model, validation)
qlora_scores = evaluate_outputs([row['gold'] for row in validation], [item['response'] for item in qlora_outputs])
qlora_record = {
    'model_name': MODEL_NAME, 'model_revision': MODEL_COMMIT, 'adapter_path': str(adapter_dir),
    'decoding': {'do_sample': False, 'max_new_tokens': MAX_NEW_TOKENS},
    'mean_latency_seconds': sum(item['latency_seconds'] for item in qlora_outputs) / len(qlora_outputs),
    'scores': qlora_scores, 'outputs': qlora_outputs,
}
with mlflow.start_run(run_name='smollm3-qlora-validation') as run:
    qlora_record['mlflow_run_id'] = run.info.run_id
    qlora_payload = json.dumps(qlora_record, indent=2, ensure_ascii=False, default=json_default)
    save_json(OUTPUT / 'qlora_validation_predictions.json', qlora_record)
    mlflow.log_params({'model_name': MODEL_NAME, 'model_revision': MODEL_COMMIT, 'adapter_run_id': training_metadata['mlflow_run_id']})
    mlflow.log_metrics({'schema_validity_rate': qlora_scores['strict']['schema_validity_rate'], 'mean_latency_seconds': qlora_record['mean_latency_seconds']})
    mlflow.log_text(qlora_payload, 'validation_predictions.json')
print({'schema_validity': qlora_scores['strict']['schema_validity_rate'], 'mean_latency_seconds': qlora_record['mean_latency_seconds'], 'mlflow_run_id': qlora_record['mlflow_run_id']})
shutil.make_archive('/kaggle/working/civicstruct_results', 'zip', root_dir=str(OUTPUT))
print('/kaggle/working/civicstruct_results.zip ready')

{'schema_validity': 0.98, 'mean_latency_seconds': 3.1866606498600074, 'mlflow_run_id': 'b8ce5a49a1bd43f99f30e3b182e7fe22'}
/kaggle/working/civicstruct_results.zip ready
